# Preprocessing berita CNN Indonesia (adaptasi dari versi CNBC)

Alur, urutan tahap, threshold, dan parameter model **tidak diubah**. Perubahan hanya pada komponen yang bergantung bahasa/skema:

| Tahap | Versi CNBC (Inggris) | Versi ini (Indonesia) |
|---|---|---|
| Skema kolom | `full_text`, `preview_summary`, `published_raw`, `section`, `keyword_matched` | dibuat dari `content`, `description`, `date`, URL; `keyword_matched` kosong |
| Simbol mata uang | percent, dollar | persen, dolar |
| Filter bahasa | `en` | `id` |
| Stopword | NLTK english + custom | NLTK indonesian + custom (padanan) |
| Lemmatization | WordNet + POS tag | Stemmer Sastrawi (nama fungsi/kolom sama) |
| Sentimen | VADER + Loughran-McDonald | InSet (VADER dan LM hanya untuk Bahasa Inggris, LM tidak punya padanan) |
| Kata kunci GPR | daftar Inggris | terjemahan satu-satu, jumlah dan urutan sama |
| Ekstraksi TF-IDF + SVD, agregasi harian, merge, cutoff 15:00 | | sama |

**Prasyarat tambahan:** file `positive.tsv` dan `negative.tsv` dari lexicon InSet di `../data/external/inset/`.


In [1]:
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [2]:
news_raw = pd.read_csv('../data/raw/cnn_internasional.csv')
bi_raw = pd.read_csv('../data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (9990, 7)
Kurs USD BI       : (1202, 5)


,title,date,author,description,image,url,content
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",2026-11-24T18:00:08+07:00,NaN,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,https://akcdn.detik.net.id/visual/2026/11/24/pm-malaysia-anwar-ibrahim-pm-jepang-sanae-takaichi-1795508133582_169.jp...,https://www.cnnindonesia.com/internasional/20261124145552-113-1430863/viral-momen-bercanda-pm-takaichi-anwar-netizen...,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",2026-11-24T15:15:05+07:00,NaN,Pemerintah Turki membantah pengumuman Malaysia yang menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke ne...,https://akcdn.detik.net.id/visual/2026/07/28/pm-malaysia-perintahkan-tindak-tegas-pengungsi-rohingya-tanpa-izin-1785...,https://www.cnnindonesia.com/internasional/20261124143121-106-1430841/malaysia-mau-kirim-10000-pengungsi-rohingya-ke...,Malaysia menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke Turki .\r\nPemerintah Turki langsung menolak ...
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,2026-11-24T13:05:05+07:00,NaN,PM Anwar mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar bulan depan.,https://akcdn.detik.net.id/visual/2025/11/05/anwar-ibrahim-1762331055170_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124114805-106-1430750/malaysia-bakal-pulangkan-6800-migran-rohingya-...,Perdana Menteri Anwar Ibrahim mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar dalam ...
3,"Setara 1 Resimen, Ribuan Tentara Rusia Bunuh Diri Imbas Stres Perang",2026-11-24T17:25:26+07:00,NaN,Sebanyak 1.087 tentara Rusia dilaporkan meninggal dunia akibat bunuh diri karena stres perang sejak Moskow melancark...,https://akcdn.detik.net.id/visual/2022/11/04/tentara-rusia-bebas-dari-tahanan-ukraina-1_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124153911-134-1430906/setara-1-resimen-ribuan-tentara-rusia-bunuh-di...,Sebanyak 1.087 tentara Rusia dilaporkan tercatat meninggal dunia akibat bunuh diri karena stres perang sejak Moskow ...
4,"Estonia Tutup Perbatasan, Rusia Ancam Negara NATO Itu Perang Terbuka",2026-11-24T16:51:04+07:00,NaN,Estonia memutuskan menutup perbatasannya dengan Rusia di tengah perang Moskow melawan Ukraina yang belum kunjung usai.,https://akcdn.detik.net.id/visual/2026/11/24/perbatasan-estonia-rusia-1795513821122_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124163928-134-1430938/estonia-tutup-perbatasan-rusia-ancam-negara-na...,Estonia memutuskan menutup perbatasannya dengan Rusia di tengah perang Moskow melawan Ukraina yang belum kunjung usa...


In [3]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9990 entries, 0 to 9989
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        9990 non-null   object
 1   date         8819 non-null   object
 2   author       2058 non-null   object
 3   description  9990 non-null   object
 4   image        9990 non-null   object
 5   url          9990 non-null   object
 6   content      9990 non-null   object
dtypes: object(7)
memory usage: 546.5+ KB


In [4]:
# Parse timestamp CNN Indonesia (sudah +07:00 / WIB)
news_raw["date_dt"] = pd.to_datetime(news_raw["date"], errors="coerce")
news_raw["date_dt_wib"] = news_raw["date_dt"].dt.tz_localize(None)

news_raw["tanggal"] = news_raw["date_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_dt_wib"].dt.strftime("%I:%M %p")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_dt_wib"]].head(5)

Kolom Berita Geopolitik:


,title,tanggal,jam,date_dt_wib
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",11/24/2026,06:00 PM,2026-11-24 18:00:08
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",11/24/2026,03:15 PM,2026-11-24 15:15:05
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,11/24/2026,01:05 PM,2026-11-24 13:05:05
3,"Setara 1 Resimen, Ribuan Tentara Rusia Bunuh Diri Imbas Stres Perang",11/24/2026,05:25 PM,2026-11-24 17:25:26
4,"Estonia Tutup Perbatasan, Rusia Ancam Negara NATO Itu Perang Terbuka",11/24/2026,04:51 PM,2026-11-24 16:51:04


In [5]:
# Penyesuaian skema dataset CNN Indonesia ke nama kolom yang dipakai pipeline.
# Tahap-tahap di bawah tidak diubah; hanya nama kolomnya yang disamakan.
news_raw["preview_summary"] = news_raw["description"]
news_raw["full_text"] = news_raw["content"]
news_raw["published_raw"] = news_raw["date"]
news_raw["section"] = news_raw["url"].str.extract(r"cnnindonesia\.com/([^/]+)/", expand=False)
news_raw["keyword_matched"] = np.nan  # tidak ada di dataset CNN; filter relevansi keyword sudah dihapus (sumber sudah internasional)

print("Baris tanpa tanggal (akan terbuang oleh filter rentang):", news_raw["date_dt"].isna().sum())
print("Rentang tanggal berita:", news_raw["date_dt_wib"].min(), "-", news_raw["date_dt_wib"].max())
news_raw[["title", "preview_summary", "full_text", "published_raw", "section"]].head(3)


Baris tanpa tanggal (akan terbuang oleh filter rentang): 1171
Rentang tanggal berita: 2026-01-22 14:15:35 - 2026-11-24 18:00:08


,title,preview_summary,full_text,published_raw,section
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,2026-11-24T18:00:08+07:00,internasional
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",Pemerintah Turki membantah pengumuman Malaysia yang menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke ne...,Malaysia menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke Turki .\r\nPemerintah Turki langsung menolak ...,2026-11-24T15:15:05+07:00,internasional
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,PM Anwar mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar bulan depan.,Perdana Menteri Anwar Ibrahim mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar dalam ...,2026-11-24T13:05:05+07:00,internasional


In [6]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning

## 1. Filter Dataset


In [7]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_dt_wib"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 9990
Setelah filter rentang 1 September 2021 - 1 September 2026 : 6346
Setelah dedup url                  : 6346
Setelah dedup title+date_raw       : 6346


In [8]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "description", "content"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 6346
Setelah drop video/gambar/audio      : 5798


,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,jam,preview_summary,full_text,published_raw,section,keyword_matched
2827,Bos Bank of America Erin Piacenti Tewas Ditusuk di Times Square AS,2026-09-01T23:43:46+07:00,Aljazeera,Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square.,https://akcdn.detik.net.id/visual/2020/06/22/ilustrasi-penusukan-2_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901233948-134-1398948/bos-bank-of-america-erin-piacenti-tewas-ditusu...,"Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square, New Yo...",2026-09-01 23:43:46+07:00,2026-09-01 23:43:46,09/01/2026,11:43 PM,Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square.,"Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square, New Yo...",2026-09-01T23:43:46+07:00,internasional,NaN
2831,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia",2026-09-01T20:40:29+07:00,NaN,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.",https://akcdn.detik.net.id/visual/2026/09/01/sco-summitkyrgyzstan-1788269721227_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901202732-120-1398902/presiden-iran-bertemu-putin-ucap-terima-kasih-...,"Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01 20:40:29+07:00,2026-09-01 20:40:29,09/01/2026,08:40 PM,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.","Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01T20:40:29+07:00,internasional,NaN
2833,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang",2026-09-01T20:00:02+07:00,NaN,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,https://akcdn.detik.net.id/visual/2023/05/30/ilustrasi-penangkapan-3_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901180103-113-1398848/kasus-jeju-polisi-korsel-dicokok-gegara-palsuk...,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01 20:00:02+07:00,2026-09-01 20:00:02,09/01/2026,08:00 PM,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01T20:00:02+07:00,internasional,NaN
2834,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru",2026-09-01T19:45:45+07:00,NaN,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,https://akcdn.detik.net.id/visual/2019/06/26/3e6d3647-1737-466a-9b93-e8bf53cb3325_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901173114-134-1398843/perang-lawan-as-iran-klaim-kembangkan-sistem-p...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01 19:45:45+07:00,2026-09-01 19:45:45,09/01/2026,07:45 PM,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01T19:45:45+07:00,internasional,NaN
2835,Kedubes Rusia Buka Suara soal Dugaan 8 WNI Jadi Tentara Bayaran,2026-09-01T19:23:24+07:00,NaN,Dubes Rusia di Indonesia pastikan pihaknya tidak melakukan rekrutmen dalam bentuk apapun untuk Angkatan Bersenjata R...,https://akcdn.detik.net.id/visual/2024/10/2

In [9]:
# Panjang text minimal
# Threshold sengaja BELUM diubah (masih 350): lihat distribusi di output, lalu tentukan nilai MIN_WORDS.
MIN_WORDS = 150

effective_text = news["content"].fillna(news["description"])
wc = effective_text.apply(word_count)

print("Distribusi jumlah kata (sebelum filter panjang):")
print(wc.describe(percentiles=[.1, .25, .5, .75, .9]).round(1))
print("\nJumlah baris yang lolos per threshold:")
for t in [50, 100, 150, 200, 250, 300, 350]:
    print(f"  >= {t:>3} kata : {(wc >= t).sum():>5} baris")

news = news.loc[wc >= MIN_WORDS]
print(f"\nSetelah filter panjang teks min {MIN_WORDS} : {len(news)}")

Distribusi jumlah kata (sebelum filter panjang):
count    5798.0
mean      326.2
std       151.9
min         9.0
10%       198.0
25%       238.0
50%       294.0
75%       374.0
90%       494.0
max      2241.0
Name: content, dtype: float64

Jumlah baris yang lolos per threshold:
  >=  50 kata :  5776 baris
  >= 100 kata :  5760 baris
  >= 150 kata :  5661 baris
  >= 200 kata :  5181 baris
  >= 250 kata :  4030 baris
  >= 300 kata :  2749 baris
  >= 350 kata :  1818 baris

Setelah filter panjang teks min 150 : 5661


In [10]:
# Filter relevansi keyword DIHAPUS: sumber data sudah cnn_internasional.csv (semua berita internasional),
# jadi filter 14 kata kunci hanya membuang berita relevan (1.818 -> 608 pada versi sebelumnya).
print(f"Setelah (tanpa) filter relevansi kata kunci : {len(news)}")

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

# Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")
print("Distribusi section:", news["section"].value_counts().to_dict())

news.head(3)

Setelah (tanpa) filter relevansi kata kunci : 5661
Setelah filter kategori/section     : 5661
Distribusi section: {'internasional': 5661}


,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,jam,preview_summary,full_text,published_raw,section,keyword_matched
0,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia",2026-09-01T20:40:29+07:00,NaN,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.",https://akcdn.detik.net.id/visual/2026/09/01/sco-summitkyrgyzstan-1788269721227_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901202732-120-1398902/presiden-iran-bertemu-putin-ucap-terima-kasih-...,"Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01 20:40:29+07:00,2026-09-01 20:40:29,09/01/2026,08:40 PM,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.","Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01T20:40:29+07:00,internasional,NaN
1,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang",2026-09-01T20:00:02+07:00,NaN,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,https://akcdn.detik.net.id/visual/2023/05/30/ilustrasi-penangkapan-3_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901180103-113-1398848/kasus-jeju-polisi-korsel-dicokok-gegara-palsuk...,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01 20:00:02+07:00,2026-09-01 20:00:02,09/01/2026,08:00 PM,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01T20:00:02+07:00,internasional,NaN
2,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru",2026-09-01T19:45:45+07:00,NaN,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,https://akcdn.detik.net.id/visual/2019/06/26/3e6d3647-1737-466a-9b93-e8bf53cb3325_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901173114-134-1398843/perang-lawan-as-iran-klaim-kembangkan-sistem-p...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01 19:45:45+07:00,2026-09-01 19:45:45,09/01/2026,07:45 PM,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01T19:45:45+07:00,internasional,NaN


In [11]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 5661
Sisa NaN pada full_text: 0


## 2. Cleaning Data News

In [12]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 5661


,title,combined_raw
0,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia","Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia Presiden Iran Bertemu Putin, Ucap Terima Kasih atas ..."
1,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang","Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang Kasus Jeju, Polisi Korsel Dicokok Gegara Pals..."
2,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru","Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru Perang Lawan AS, Iran Klaim Kembangkan Sistem..."


In [13]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 33


,simbol,nama_unicode,jumlah
0,",",COMMA,118893
1,.,FULL STOP,113990
2,"""",QUOTATION MARK,41268
3,-,HYPHEN-MINUS,27348
4,(,LEFT PARENTHESIS,17087
5,),RIGHT PARENTHESIS,17084
6,/,SOLIDUS,8339
7,',APOSTROPHE,3865
8,:,COLON,2875
9,?,QUESTION MARK,1530


In [14]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "persen",
    "$": "dolar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia Presiden Iran Bertemu Putin, Ucap Terima Kasih atas ...",Presiden Iran Bertemu Putin Ucap Terima Kasih atas Sikap Rusia Presiden Iran Bertemu Putin Ucap Terima Kasih atas Si...
1,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang Kasus Jeju, Polisi Korsel Dicokok Gegara Pals...",Kasus Jeju Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang Kasus Jeju Polisi Korsel Dicokok Gegara Palsuk...
2,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru Perang Lawan AS, Iran Klaim Kembangkan Sistem...",Perang Lawan AS Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru Perang Lawan AS Iran Klaim Kembangkan Sistem P...


In [15]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Presiden Iran Bertemu Putin Ucap Terima Kasih atas Sikap Rusia Presiden Iran Bertemu Putin Ucap Terima Kasih atas Si...,presiden iran bertemu putin ucap terima kasih atas sikap rusia presiden iran bertemu putin ucap terima kasih atas si...
1,Kasus Jeju Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang Kasus Jeju Polisi Korsel Dicokok Gegara Palsuk...,kasus jeju polisi korsel dicokok gegara palsukan laporan orang hilang kasus jeju polisi korsel dicokok gegara palsuk...
2,Perang Lawan AS Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru Perang Lawan AS Iran Klaim Kembangkan Sistem P...,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...


In [16]:
#memastikan bahasanya indonesia semua
TARGET_LANG = "id"
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == TARGET_LANG].reset_index(drop=True)
print(f"\nSetelah filter bahasa Indonesia (id): {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
id    5661
Name: count, dtype: int64

Setelah filter bahasa Indonesia (id): 5661 (dibuang 0 baris)


,lower_text,detected_lang
0,presiden iran bertemu putin ucap terima kasih atas sikap rusia presiden iran bertemu putin ucap terima kasih atas si...,id
1,kasus jeju polisi korsel dicokok gegara palsukan laporan orang hilang kasus jeju polisi korsel dicokok gegara palsuk...,id
2,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...,id


## 3. Cleaning Data Kurs

In [17]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [18]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [19]:
#stopword removal
# Padanan CUSTOM_STOPWORDS versi CNBC: nama sumber, kata ajakan baca, kata kerja pelaporan (said/says), angka kata.
CUSTOM_STOPWORDS = {
    "cnn", "indonesia", "cnnindonesia", "artikel", "baca", "juga",
    "dilansir", "dikutip", "kata", "katanya", "mengatakan", "ujar", "ujarnya",
    "ucap", "ungkap", "tutur", "satu", "dua",
}
STOPWORDS_ID = set(stopwords.words("indonesian")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Indonesia & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_ID and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 375.4
Rata-rata jumlah kata sesudah stopword removal  : 228.0


,lower_text,text_no_stopwords
0,presiden iran bertemu putin ucap terima kasih atas sikap rusia presiden iran bertemu putin ucap terima kasih atas si...,presiden iran bertemu putin terima kasih sikap rusia presiden iran bertemu putin terima kasih sikap rusia bertemu kt...
1,kasus jeju polisi korsel dicokok gegara palsukan laporan orang hilang kasus jeju polisi korsel dicokok gegara palsuk...,jeju polisi korsel dicokok gegara palsukan laporan orang hilang jeju polisi korsel dicokok gegara palsukan laporan o...
2,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...


In [20]:
#lemmatization -> stemming Bahasa Indonesia (Sastrawi)
# WordNet + POS tagger hanya untuk Bahasa Inggris. Untuk Bahasa Indonesia padanannya adalah stemmer Sastrawi.
# Nama fungsi/kolom (lemmatize_tokens, tokens_lemmatized, text_lemmatized) sengaja dipertahankan
# supaya seluruh tahap setelahnya tidak berubah.
stemmer = StemmerFactory().create_stemmer()
_stem_cache = {}


def stem_token(token: str) -> str:
    # Cache per kata unik: Sastrawi lambat bila dipanggil berulang untuk kata yang sama.
    if token not in _stem_cache:
        _stem_cache[token] = stemmer.stem(token)
    return _stem_cache[token]


def lemmatize_tokens(tokens: list) -> list:
    if not tokens:
        return []
    return [stem_token(token) for token in tokens]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,presiden iran bertemu putin terima kasih sikap rusia presiden iran bertemu putin terima kasih sikap rusia bertemu kt...,presiden iran temu putin terima kasih sikap rusia presiden iran temu putin terima kasih sikap rusia temu ktt sco kyr...
1,jeju polisi korsel dicokok gegara palsukan laporan orang hilang jeju polisi korsel dicokok gegara palsukan laporan o...,jeju polisi korsel cokok gegara palsu lapor orang hilang jeju polisi korsel cokok gegara palsu lapor orang hilang po...
2,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...,perang lawan as iran klaim kembang sistem tahan udara baru perang lawan as iran klaim kembang sistem tahan udara bar...


# Penyelarasan Data

In [21]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

# versi naive WIB (tanpa tz) untuk alignment & merge, sama seperti Tanggal_dt di data kurs
news['date_raw_dt'] = news['published_raw'].dt.tz_localize(None)


In [22]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [23]:
def align_to_trading_date(dt):
    # Step 1: kalau lewat cutoff 15:15:00, geser ke hari berikutnya
    cutoff = dt.replace(hour=15, minute=0, second=0, microsecond=0)
    if dt > cutoff:
        dt = dt + pd.Timedelta(days=1)

    # Step 2: kalau hasilnya jatuh di weekend, geser ke Senin
    if dt.weekday() == 5:      # Sabtu
        dt = dt + pd.Timedelta(days=2)
    elif dt.weekday() == 6:    # Minggu
        dt = dt + pd.Timedelta(days=1)

    return dt.normalize()  # buang jam, sisakan tanggal saja

news['aligned_trading_date'] = news['date_raw_dt'].apply(align_to_trading_date)
news

,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,...,normalized_text,lower_text,detected_lang,tokens_no_stopwords,text_no_stopwords,tokens_lemmatized,text_lemmatized,time,date_raw_dt,aligned_trading_date
0,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia",2026-09-01,NaN,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.",https://akcdn.detik.net.id/visual/2026/09/01/sco-summitkyrgyzstan-1788269721227_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901202732-120-1398902/presiden-iran-bertemu-putin-ucap-terima-kasih-...,"Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01 20:40:29+07:00,2026-09-01 20:40:29,09/01/2026,...,Presiden Iran Bertemu Putin Ucap Terima Kasih atas Sikap Rusia Presiden Iran Bertemu Putin Ucap Terima Kasih atas Si...,presiden iran bertemu putin ucap terima kasih atas sikap rusia presiden iran bertemu putin ucap terima kasih atas si...,id,"[presiden, iran, bertemu, putin, terima, kasih, sikap, rusia, presiden, iran, bertemu, putin, terima, kasih, sikap, ...",presiden iran bertemu putin terima kasih sikap rusia presiden iran bertemu putin terima kasih sikap rusia bertemu kt...,"[presiden, iran, temu, putin, terima, kasih, sikap, rusia, presiden, iran, temu, putin, terima, kasih, sikap, rusia,...",presiden iran temu putin terima kasih sikap rusia presiden iran temu putin terima kasih sikap rusia temu ktt sco kyr...,20:40:29,2026-09-01 20:40:29,2026-09-02
1,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang",2026-09-01,NaN,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,https://akcdn.detik.net.id/visual/2023/05/30/ilustrasi-penangkapan-3_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901180103-113-1398848/kasus-jeju-polisi-korsel-dicokok-gegara-palsuk...,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01 20:00:02+07:00,2026-09-01 20:00:02,09/01/2026,...,Kasus Jeju Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang Kasus Jeju Polisi Korsel Dicokok Gegara Palsuk...,kasus jeju polisi korsel dicokok gegara palsukan laporan orang hilang kasus jeju polisi korsel dicokok gegara palsuk...,id,"[jeju, polisi, korsel, dicokok, gegara, palsukan, laporan, orang, hilang, jeju, polisi, korsel, dicokok, gegara, pal...",jeju polisi korsel dicokok gegara palsukan laporan orang hilang jeju polisi korsel dicokok gegara palsukan laporan o...,"[jeju, polisi, korsel, cokok, gegara, palsu, lapor, orang, hilang, jeju, polisi, korsel, cokok, gegara, palsu, lapor...",jeju polisi korsel cokok gegara palsu lapor orang hilang jeju polisi korsel cokok gegara palsu lapor orang hilang po...,20:00:02,2026-09-01 20:00:02,2026-09-02
2,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru",2026-09-01,NaN,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,https://akcdn.detik.net.id/visual/2019/06/26/3e6d3647-1737-466a-9b93-e8bf53cb3325_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901173114-134-1398843/perang-lawan-as-iran-klaim-kembangkan-sistem-p...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01 19:45:45+07:00,2026-09-01 19:45:45,09/01/2026,...,Perang Lawan AS Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru Perang Lawan AS Iran Klaim Kembangkan Sistem P...,perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru perang lawan as iran klaim kembangkan sistem p...,id,"[perang, lawan, as, iran, klaim, kembangkan, sistem, pertahanan, udara, terbaru, perang, lawan, as, iran, klaim, kem...",perang lawan as iran klaim kembangkan sistem pertahanan udara terbaru per

In [24]:
# Geser aligned_trading_date ke hari kurs BERIKUTNYA yang tersedia (merge_asof, forward).
# Sebelumnya berita yang jatuh di hari tanpa kurs (libur nasional, dll.) hilang saat merge.
news["aligned_trading_date_raw"] = news["aligned_trading_date"]
news["_pos"] = np.arange(len(news))

kurs_dates = (
    bi_raw[["Tanggal_dt"]].drop_duplicates().sort_values("Tanggal_dt")
    .rename(columns={"Tanggal_dt": "kurs_date"})
)

news = pd.merge_asof(
    news.sort_values("aligned_trading_date_raw").drop(columns=["aligned_trading_date"]),
    kurs_dates,
    left_on="aligned_trading_date_raw",
    right_on="kurs_date",
    direction="forward",
).rename(columns={"kurs_date": "aligned_trading_date"})

n_geser = (news["aligned_trading_date"] != news["aligned_trading_date_raw"]).sum()
n_tanpa_kurs = news["aligned_trading_date"].isna().sum()
print(f"Berita yang digeser ke hari kurs berikutnya : {n_geser}")
print(f"Berita tanpa hari kurs berikutnya (dibuang) : {n_tanpa_kurs}  (jatuh setelah tanggal kurs terakhir)")

news = (
    news.dropna(subset=["aligned_trading_date"])
    .sort_values("_pos").drop(columns="_pos")
    .reset_index(drop=True)
)
print(f"Jumlah berita setelah penyelarasan ke kurs   : {len(news)}")

Berita yang digeser ke hari kurs berikutnya : 544
Berita tanpa hari kurs berikutnya (dibuang) : 10  (jatuh setelah tanggal kurs terakhir)
Jumlah berita setelah penyelarasan ke kurs   : 5651


In [25]:
daily_news = news.groupby('aligned_trading_date').agg(
    news_count=('title', 'count'),
    all_titles=('title', lambda x: ' | '.join(x)),
    all_text_lemmatized=('text_lemmatized', lambda x: ' | '.join(x))
).reset_index()

In [26]:
merged_data = pd.merge(
    bi_raw,
    daily_news,
    left_on='Tanggal_dt',                   
    right_on='aligned_trading_date',   
    how='left'
)

merged_data

,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,news_count,all_titles,all_text_lemmatized
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,27.0,"Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida | Profil Dan Driscoll, Menteri Angkatan Darat ...",eks pm keluh anak muda gen as israel genosida eks pm keluh anak muda gen as israel genosida mantan perdana menteri i...
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,72.0,"Intelijen Ukraina Klaim 8 WNI Jadi Tentara Bayaran Rusia | RI, AS & Sekutu Gelar Latihan Militer Super Garuda Shield...",intelijen ukraina klaim wni tentara bayar rusia intelijen ukraina klaim wni tentara bayar rusia dinas intelijen nege...
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,32.0,"Presiden Timor Leste Temui Prabowo di Istana Hari Ini | Raja Norwegia Harald V Wafat, Akhir Masa Penguasa Negeri Vik...",presiden timor leste temu prabowo istana presiden timor leste temu prabowo istana presiden prabowo subianto temu pre...
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,2026-08-27,32.0,165 Orang Tewas & 1.384 Hilang Imbas Banjir Bandang di Nepal-Tibet | Eks PM Malaysia Didakwa UU Anti-Korupsi Gegara ...,165 orang tewas 384 hilang imbas banjir bandang nepal tibet 165 orang tewas 384 hilang imbas banjir bandang nepal ti...
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,47.0,Korsel Kirim Jet Tempur Awasi Pesawat Rusia Masuk Zona Pertahanan | Siapa Profesor Malaysia yang Klaim Orang Melayu ...,korsel kirim jet tempur awas pesawat rusia masuk zona tahan korsel kirim jet tempur awas pesawat rusia masuk zona ta...
...,...,...,...,...,...,...,...,...,...,...,...,...
1197,1198,1,14310.20,14167.81,2021-09-07,09/07/2021,12:00 AM,2021-09-07,NaT,NaN,NaN,NaN
1198,1199,1,14332.31,14189.70,2021-09-06,09/06/2021,12:00 AM,2021-09-06,NaT,NaN,NaN,NaN
1199,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03,NaT,NaN,NaN,NaN
1200,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02,NaT,NaN,NaN,NaN


In [27]:
# --- CEK HASIL CUTOFF (pakai date_raw_dt yang sudah WIB) ---
news['jam'] = news['date_raw_dt'].dt.time
news['weekday'] = news['date_raw_dt'].dt.day_name()
news['lewat_cutoff'] = news['date_raw_dt'].dt.time > pd.Timestamp("15:15:00").time()

# Tampilkan berita yang persis di sekitar jam cutoff (14:50 - 15:30) untuk dicek manual
mask_sekitar_cutoff = news['jam'].apply(lambda t: (t.hour == 15 and t.minute <= 30) or (t.hour == 14 and t.minute >= 50))
cek_cutoff = news.loc[mask_sekitar_cutoff, ['title', 'date_raw_dt', 'jam', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_cutoff.sort_values('date_raw_dt').to_string(index=False))

# Cek kasus weekend shifting (berita Jumat setelah cutoff -> harus jadi Senin)
mask_weekend_case = news['weekday'].isin(['Friday', 'Saturday', 'Sunday'])
cek_weekend = news.loc[mask_weekend_case, ['title', 'date_raw_dt', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_weekend.sort_values('date_raw_dt').to_string(index=False))

                                                                 title         date_raw_dt      jam   weekday  lewat_cutoff aligned_trading_date
     Rusia, Ukraina, AS Gelar Pertemuan di Abu Dhabi, Perang Berakhir? 2026-01-23 15:05:06 15:05:06    Friday         False           2026-01-26
                           Remaja 12 Tahun di Sydney Tewas Digigit Hiu 2026-01-24 15:16:23 15:16:23  Saturday          True           2026-01-26
                              AS Resmi Keluar dari WHO, Apa Alasannya? 2026-01-25 15:00:53 15:00:53    Sunday         False           2026-01-26
   Demo Penembakan Petugas ICE Meluas, Trump Kirim Utusan ke Minnesota 2026-01-27 14:58:52 14:58:52   Tuesday         False           2026-01-27
 Kim Jong Un Perkuat Senjata Nuklir Korut: Musuh Akan Sangat Menderita 2026-01-28 15:00:41 15:00:41 Wednesday         False           2026-01-29
 Wakil Kepala Menteri Wilayah di India Tewas Imbas Pesawat Dinas Jatuh 2026-01-28 15:12:42 15:12:42 Wednesday         False       

In [28]:
tanggal_dicek = cek_cutoff['aligned_trading_date'].unique()

for tgl in tanggal_dicek:
    subset = merged_data[merged_data['aligned_trading_date'] == tgl]
    print(f"\n=== Trading date: {tgl.date()} ===")
    print(f"Jumlah berita ter-merge: {subset['news_count'].values}")
    # tampilkan judul2 berita yg masuk ke tanggal ini dari df news asli
    judul_masuk = news[news['aligned_trading_date'] == tgl][['title', 'published_raw']]
    print(judul_masuk.sort_values('published_raw').to_string(index=False))


=== Trading date: 2026-09-01 ===
Jumlah berita ter-merge: [27.]
                                                                 title             published_raw
      Update Banjir Bandang Nepal-China: 919 Orang Tewas, 4.247 Hilang 2026-08-31 15:15:00+07:00
   MK Tolak Usulan Ganti Nama Laut Cina Selatan Jadi Laut Natuna Utara 2026-08-31 15:45:00+07:00
                        Daftar Negara Bantu Indonesia Tangani Karhutla 2026-08-31 16:15:30+07:00
 Kemlu RI Verifikasi Informasi Dugaan 8 WNI Jadi Tentara Bayaran Rusia 2026-08-31 16:51:51+07:00
        Masa Bodoh Ocehan Trump, Korut Tegaskan Tak Akan Denuklirisasi 2026-08-31 18:15:47+07:00
   Kemlu RI Telusuri WNI Diduga Jadi Korban TPPO di Jeju Korea Selatan 2026-08-31 18:47:38+07:00
   Menlu Iran Cap Netanyahu Ular, Tipu Muslihat ke AS Biar Ikut Perang 2026-08-31 19:34:37+07:00
  Terdampak Karhutla RI, Kualitas Udara Malaysia Masuk Kategori Bahaya 2026-08-31 20:55:00+07:00
 Ukraina Klaim 8 WNI Tentara Bayaran Rusia sampai WNI Korban T

# Ekstraksi Fitur NLP


In [29]:
from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Sentimen berbasis Lexicon

In [30]:
# Lexicon sentimen Bahasa Indonesia: InSet (Koto & Rahmaningtyas, 2017)
# Pengganti VADER (general-purpose) dan Loughran-McDonald (finance-specific) yang hanya untuk Bahasa Inggris.
# Letakkan positive.tsv dan negative.tsv di folder berikut (kolom: word, weight).
INSET_DIR = Path("../data/external/inset")
INSET_NEGATIONS = {"tidak", "tak", "bukan", "belum", "jangan", "tanpa"}


def _load_inset_lexicon(inset_dir: Path = INSET_DIR) -> dict:
    lexicon = {}
    for fname, sign in (("positive.tsv", 1), ("negative.tsv", -1)):
        path = Path(inset_dir) / fname
        if not path.exists():
            raise FileNotFoundError(f"File lexicon InSet tidak ditemukan: {path}")
        tmp = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1], names=["word", "weight"], dtype=str)
        tmp["weight"] = pd.to_numeric(tmp["weight"], errors="coerce")
        tmp = tmp.dropna()  # baris header ikut terbuang di sini
        for word, weight in zip(tmp["word"].str.strip().str.lower(), tmp["weight"].abs()):
            lexicon[word] = sign * float(weight)
    return lexicon


_inset = _load_inset_lexicon()


def compute_inset_sentiment(df: pd.DataFrame, text_col: str = "lower_text") -> pd.DataFrame:
    # Skor bag-of-words berbobot; kata sentimen yang didahului negasi dibalik tanda (setara negasi pada VADER).
    def _score(text: str) -> pd.Series:
        tokens = text.split() if isinstance(text, str) else []
        pos = neg = 0.0
        for i, tok in enumerate(tokens):
            w = _inset.get(tok)
            if w is None:
                continue
            if i > 0 and tokens[i - 1] in INSET_NEGATIONS:
                w = -w
            if w > 0:
                pos += w
            else:
                neg += -w
        n_tokens = max(len(tokens), 1)
        total = pos - neg
        return pd.Series({
            "inset_pos": pos / n_tokens,
            "inset_neg": neg / n_tokens,
            "inset_compound": total / np.sqrt(total ** 2 + 15),     # normalisasi ala VADER (alpha=15)
            "inset_polarity": total / (pos + neg) if (pos + neg) > 0 else 0.0,  # (P-N)/(P+N) ala LM polarity
        })

    scores = df[text_col].fillna("").apply(_score)
    return pd.concat([df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)


# 3. Indeks risiko geopolitik berbasis keyword density 
# Terjemahan satu-satu dari daftar versi Inggris (jumlah dan urutan kata kunci sama)
DEFAULT_GPR_KEYWORDS = [
    "geopolitik", "risiko geopolitik", "ketegangan geopolitik",
    "fragmentasi geopolitik", "konflik bersenjata", "konflik global",
    "konflik internasional", "ketegangan militer", "keamanan nasional",
    "sanksi", "perang dagang", "tarif", "embargo", "kontrol ekspor",
    "proteksionisme", "gangguan rantai pasok", "diplomasi",
    "kebijakan luar negeri", "nato", "brics", "opec", "g7",
]

def compute_gpr_density(
    df: pd.DataFrame,
    text_col: str = "lower_text",
    keywords: list[str] | None = None,
) -> pd.DataFrame:

    kw = keywords or DEFAULT_GPR_KEYWORDS
    pattern = re.compile("|".join(re.escape(k) for k in kw))

    def _count(text: str) -> int:
        if not isinstance(text, str) or not text:
            return 0
        return len(pattern.findall(text))

    hits = df[text_col].apply(_count)
    word_count = df[text_col].fillna("").str.split().apply(len).replace(0, 1)

    out = df.copy()
    out["gpr_hits"] = hits
    out["gpr_density"] = hits / word_count
    return out

# 4. Vectorization klasik: TF-IDF + SVD (LSA) untuk reduksi dimensi
@dataclass
class TfidfSvdExtractor:
    max_features: int = 5000
    n_components: int = 75
    ngram_range: tuple[int, int] = (1, 2)
    min_df: int = 5
    random_state: int = 42

    _vectorizer: TfidfVectorizer = field(init=False, default=None, repr=False)
    _svd: TruncatedSVD = field(init=False, default=None, repr=False)
    _is_fitted: bool = field(init=False, default=False, repr=False)

    def fit(self, texts: pd.Series) -> "TfidfSvdExtractor":
        self._vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            min_df=self.min_df,
            sublinear_tf=True,
        )
        tfidf_matrix = self._vectorizer.fit_transform(texts.fillna(""))

        # n_components dibatasi supaya tidak melebihi jumlah fitur/dokumen train
        n_comp = min(self.n_components, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
        self._svd = TruncatedSVD(n_components=max(n_comp, 2), random_state=self.random_state)
        self._svd.fit(tfidf_matrix)
        self._is_fitted = True
        return self

    def transform(self, texts: pd.Series) -> pd.DataFrame:
        if not self._is_fitted:
            raise RuntimeError("Panggil .fit(train_texts) dulu sebelum .transform().")
        tfidf_matrix = self._vectorizer.transform(texts.fillna(""))
        reduced = self._svd.transform(tfidf_matrix)
        cols = [f"tfidf_svd_{i}" for i in range(reduced.shape[1])]
        return pd.DataFrame(reduced, columns=cols, index=texts.index)

    @property
    def explained_variance_ratio_(self) -> np.ndarray:
        return self._svd.explained_variance_ratio_


# 5. Agregasi fitur level-artikel -> level-harian (join key: aligned_trading_date)
def aggregate_daily_features(
    article_df: pd.DataFrame,
    date_col: str = "aligned_trading_date",
) -> pd.DataFrame:
    sentiment_cols = [c for c in article_df.columns if c.startswith("inset_")]
    svd_cols = [c for c in article_df.columns if c.startswith("tfidf_svd_")]
    mean_cols = sentiment_cols + svd_cols + (["gpr_density"] if "gpr_density" in article_df else [])

    agg_dict = {c: "mean" for c in mean_cols}
    if "gpr_hits" in article_df.columns:
        agg_dict["gpr_hits"] = "sum"

    daily = article_df.groupby(date_col).agg(agg_dict)
    daily["news_count"] = article_df.groupby(date_col).size()
    return daily.reset_index()

# 1.1 Fitur Sentimen (InSet Lexicon)

In [31]:
news_feat = compute_inset_sentiment(news, text_col='text_lemmatized')

news_base = compute_inset_sentiment(news, text_col='lower_text')


In [32]:
print(f"Jumlah baris feat: {len(news_feat)}")
news_feat[['title', 'inset_compound', 'inset_polarity']].head(5)

# Hasil sentimen InSet (teks lemmatized/stem)
inset_feat_result = news_feat[
    [
        "title",
        "inset_neg",
        "inset_pos",
        "inset_compound",
        "inset_polarity",
    ]
].copy()


Jumlah baris feat: 5651


In [33]:
inset_feat_result.head(5)

,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida,0.635802,0.135802,-0.998859,-0.648000
1,"Profil Dan Driscoll, Menteri Angkatan Darat AS yang Mundur",0.793893,0.423664,-0.999204,-0.304075
2,Korban Tewas Banjir Bandang Nepal-China Tembus 1.000 Orang,1.166667,0.197531,-0.999696,-0.710407
3,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.800000,0.275000,-0.999734,-0.488372
4,China Bangun Kembali Jalan Rusak dalam Hitungan Hari Usai Banjir Parah,1.107744,0.329966,-0.999859,-0.540984


In [34]:
print(f"Jumlah baris lower: {len(news_base)}")
news_base[['title', 'inset_compound', 'inset_polarity']].head(5)

# Hasil sentimen InSet (lower_text)
inset_base_result = news_base[
    [
        "title",
        "inset_neg",
        "inset_pos",
        "inset_compound",
        "inset_polarity",
    ]
].copy()

inset_base_result.head(5)


Jumlah baris lower: 5651


,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida,0.597744,0.203008,-0.999320,-0.492958
1,"Profil Dan Driscoll, Menteri Angkatan Darat AS yang Mundur",0.840102,0.230964,-0.999870,-0.568720
2,Korban Tewas Banjir Bandang Nepal-China Tembus 1.000 Orang,1.029412,0.105042,-0.999845,-0.814815
3,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.812133,0.244618,-0.999911,-0.537037
4,China Bangun Kembali Jalan Rusak dalam Hitungan Hari Usai Banjir Parah,0.948827,0.213220,-0.999937,-0.633028


In [35]:
inset_base_result.head(5)

,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida,0.597744,0.203008,-0.999320,-0.492958
1,"Profil Dan Driscoll, Menteri Angkatan Darat AS yang Mundur",0.840102,0.230964,-0.999870,-0.568720
2,Korban Tewas Banjir Bandang Nepal-China Tembus 1.000 Orang,1.029412,0.105042,-0.999845,-0.814815
3,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.812133,0.244618,-0.999911,-0.537037
4,China Bangun Kembali Jalan Rusak dalam Hitungan Hari Usai Banjir Parah,0.948827,0.213220,-0.999937,-0.633028


In [36]:
comparison = pd.DataFrame({
    "title": news["title"],
    "inset_lower": news_base["inset_compound"],
    "inset_lemma": news_feat["inset_compound"],
    "polarity_lower": news_base["inset_polarity"],
    "polarity_lemma": news_feat["inset_polarity"],
})

comparison["inset_diff"] = comparison["inset_lower"] - comparison["inset_lemma"]
comparison["polarity_diff"] = comparison["polarity_lower"] - comparison["polarity_lemma"]

comparison.head(5)


,title,inset_lower,inset_lemma,polarity_lower,polarity_lemma,inset_diff,polarity_diff
0,Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida,-0.999320,-0.998859,-0.492958,-0.648000,-0.000462,0.155042
1,"Profil Dan Driscoll, Menteri Angkatan Darat AS yang Mundur",-0.999870,-0.999204,-0.568720,-0.304075,-0.000666,-0.264645
2,Korban Tewas Banjir Bandang Nepal-China Tembus 1.000 Orang,-0.999845,-0.999696,-0.814815,-0.710407,-0.000149,-0.104408
3,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,-0.999911,-0.999734,-0.537037,-0.488372,-0.000176,-0.048665
4,China Bangun Kembali Jalan Rusak dalam Hitungan Hari Usai Banjir Parah,-0.999937,-0.999859,-0.633028,-0.540984,-0.000078,-0.092044


In [37]:
print("Perbedaan InSet compound :", (comparison["inset_diff"] != 0).sum())
print("Perbedaan InSet polarity :", (comparison["polarity_diff"] != 0).sum())


Perbedaan InSet compound : 5624
Perbedaan InSet polarity : 5650


### Keputusan final sumber teks untuk lexicon

- **InSet -> `lower_text`**: InSet adalah kamus kata Bahasa Indonesia yang berisi bentuk dasar dan sebagian bentuk berimbuhan. Teks `lower_text` masih memuat negasi ("tidak", "bukan", "belum") yang dipakai untuk membalik tanda kata sentimen, dan negasi ini hilang pada `text_lemmatized` karena termasuk stopword. Ini sejalan dengan alasan VADER -> `lower_text` pada versi CNBC.
- Cek hasil perbandingan di atas; bila selisih lemma vs lower besar, pertimbangkan ulang sebelum dikunci.
- **Loughran-McDonald tidak punya padanan Bahasa Indonesia**, jadi fitur `lm_*` tidak ada di versi ini.


In [38]:
# Kombinasi final: InSet dari lower_text
news_feat = compute_inset_sentiment(news, text_col='lower_text')

print(f"Jumlah baris: {len(news_feat)}")
news_feat[['title', 'inset_compound', 'inset_polarity']].head(5)


Jumlah baris: 5651


,title,inset_compound,inset_polarity
0,Eks PM Mengeluh Anak Muda & Gen Z AS Hanya Ingat Israel soal Genosida,-0.999320,-0.492958
1,"Profil Dan Driscoll, Menteri Angkatan Darat AS yang Mundur",-0.999870,-0.568720
2,Korban Tewas Banjir Bandang Nepal-China Tembus 1.000 Orang,-0.999845,-0.814815
3,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,-0.999911,-0.537037
4,China Bangun Kembali Jalan Rusak dalam Hitungan Hari Usai Banjir Parah,-0.999937,-0.633028


# 1.2 Indeks Resiko Geopolitik berbasis Keyword

In [39]:
news_feat = compute_gpr_density(news_feat, text_col='lower_text', keywords=DEFAULT_GPR_KEYWORDS)

print(news_feat['gpr_hits'].describe())
news_feat[['title', 'gpr_hits', 'gpr_density']].sort_values('gpr_density', ascending=False).head(5)

count    5651.000000
mean        0.622368
std         1.858924
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        33.000000
Name: gpr_hits, dtype: float64


,title,gpr_hits,gpr_density
2708,"Hadiri FMM BRICS, RI Tegaskan Peran Aktif hingga Perkuat Kolaborasi",20,0.058140
2725,Menlu RI Sugiono Tiba di India untuk Hadiri Pertemuan BRICS,19,0.054286
5004,Drama Labil Tarif Trump: Dari 10% Naik Jadi 15% dalam Sehari,25,0.050505
4356,"Hampir 50 Tahun Embargo, Diserang AS-Israel, Kenapa Iran Masih Kuat?",28,0.046434
3961,Trump Kecewa NATO Tak Bantu Perangi Iran: Jangan Lupakan Momen Ini!,13,0.043771


# 1.3 TF-IDF + SVD

In [40]:
extractor = TfidfSvdExtractor(max_features=5000, n_components=75, min_df=5)
extractor.fit(news_feat['text_lemmatized'])

svd_vec = extractor.transform(news_feat['text_lemmatized'])
news_feat = pd.concat([news_feat.reset_index(drop=True), svd_vec.reset_index(drop=True)], axis=1)

print(f"Explained variance ratio (kumulatif, {extractor.n_components} komponen): {extractor.explained_variance_ratio_.sum():.3f}")
svd_vec.head(3)

Explained variance ratio (kumulatif, 75 komponen): 0.250


,tfidf_svd_0,tfidf_svd_1,tfidf_svd_2,tfidf_svd_3,tfidf_svd_4,tfidf_svd_5,tfidf_svd_6,tfidf_svd_7,tfidf_svd_8,tfidf_svd_9,...,tfidf_svd_65,tfidf_svd_66,tfidf_svd_67,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74
0,0.200500,-0.131902,0.036528,-0.121444,0.133110,-0.029837,0.033123,-0.153261,-0.058531,0.035336,...,0.006405,0.000451,-0.047896,0.037285,0.009917,-0.028351,0.016133,-0.015947,0.028298,-0.025461
1,0.232732,-0.047727,0.040775,0.019077,-0.079056,-0.062312,-0.082943,-0.013795,0.100379,0.038577,...,0.041954,-0.005243,0.033574,0.099536,0.076884,0.017954,0.064593,0.022870,0.037344,0.016442
2,0.141100,-0.172308,-0.104442,0.090407,-0.049353,-0.064196,0.101025,0.143106,-0.028227,-0.105512,...,0.013090,-0.002786,0.022531,0.012450,-0.029888,0.001809,-0.013901,-0.046844,0.016326,-0.000906


# 1.4 Agregaris ke Level Harian & Merge ke Data Kurs

daily_features = aggregate_daily_features(news_feat, date_col='aligned_trading_date')
print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head(5)

In [41]:
daily_features = aggregate_daily_features(
    news_feat,
    date_col="aligned_trading_date",
)

print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head()

Jumlah hari trading dengan fitur NLP: 142


,aligned_trading_date,inset_pos,inset_neg,inset_compound,inset_polarity,tfidf_svd_0,tfidf_svd_1,tfidf_svd_2,tfidf_svd_3,tfidf_svd_4,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,2026-01-22,0.285888,0.821936,-0.999750,-0.484684,0.227160,-0.002037,-0.045360,-0.059232,-0.034616,...,0.010974,0.042942,-0.041144,0.019394,-0.023900,0.028186,0.021976,0.000000,0,2
1,2026-01-23,0.295228,0.715789,-0.999425,-0.413349,0.201951,-0.115782,0.082266,-0.001451,0.028784,...,-0.005760,-0.006601,0.012954,0.043252,-0.009305,0.002044,0.016059,0.000510,4,22
2,2026-01-26,0.276287,0.773596,-0.959487,-0.466409,0.203774,-0.104822,0.022291,0.034301,-0.031271,...,0.012294,0.002330,-0.009637,0.013510,-0.011593,-0.007965,-0.001302,0.002142,43,55
3,2026-01-27,0.269114,0.730722,-0.999510,-0.458644,0.221136,-0.064585,-0.002448,0.035579,-0.023934,...,-0.002905,0.002135,-0.002962,-0.021522,-0.012237,-0.001616,0.003916,0.002971,29,25
4,2026-01-28,0.258983,0.740488,-0.999452,-0.471248,0.244725,-0.045965,0.022943,0.004784,-0.004366,...,0.001944,-0.003338,-0.034000,-0.007045,-0.018382,0.004013,-0.011572,0.001158,12,26


In [42]:
merged_data_v2 = pd.merge(
    bi_raw,
    daily_features,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
)

print(f"Baris merged_data_v2: {len(merged_data_v2)}")
print(f"Hari tanpa berita (NaN fitur NLP): {merged_data_v2['news_count'].isna().sum()}")
merged_data_v2.head(5)

Baris merged_data_v2: 1202
Hari tanpa berita (NaN fitur NLP): 1060


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,inset_pos,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,0.244561,...,0.006337,-0.004573,0.011754,0.009604,-0.007705,0.021065,-0.000481,0.000232,3.0,27.0
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,0.273158,...,0.000878,-0.002388,-0.002374,-0.000431,-0.006745,0.013679,-0.008868,0.002786,99.0,72.0
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,0.266502,...,-0.005975,-0.018662,0.009431,-0.000674,-0.015497,0.001369,-0.020493,0.004577,56.0,32.0
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,2026-08-27,0.242092,...,0.001773,-0.002727,-0.007533,0.001091,-0.014113,-0.012850,-0.011421,0.001089,11.0,32.0
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,0.248799,...,0.007153,-0.005513,0.006103,0.010232,-0.022878,0.000797,0.004135,0.002663,52.0,47.0


# 1.5 Sanity Check

In [43]:
check = merged_data_v2.sort_values('Tanggal_dt').copy()
check['kurs_next'] = check['Kurs Jual'].shift(-1)
check['kurs_change_next'] = check['kurs_next'] - check['Kurs Jual']

corr_cols = ['inset_compound', 'inset_polarity', 'gpr_density', 'gpr_hits', 'news_count', 'kurs_change_next']
check[corr_cols].corr()['kurs_change_next']

inset_compound      0.054923
inset_polarity      0.069914
gpr_density         0.126411
gpr_hits            0.113285
news_count          0.013481
kurs_change_next    1.000000
Name: kurs_change_next, dtype: float64

In [44]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "dataset_processed_cnn_bi.csv"
merged_data_v2.to_csv(output_path, index=False)

print(f"File berhasil disimpan di: {output_path.resolve()}")

File berhasil disimpan di: C:\Users\ASUS\Downloads\code\nlpproject-grestercinta\data\processed\dataset_processed_cnn_bi.csv


In [ ]:
#cek shape
merged_data_v2.shape

NameError: name 'merged_data_v2' is not defined